# 3.0 — RAG Local Persistence

This notebook loads a PDF, splits it into chunks, creates embeddings, stores them in a local Chroma vector database, and runs semantic searches.

**Before running:** make sure `Docs/fabric-onelake.pdf` exists and your `.env` file contains `OPENAI_API_KEY=your_api_key`.

In [ ]:
# ============================================================
# 3.0 - RAG Local Persistence
# ============================================================

import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma


# 1. Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found. Please check your .env file."
    )

print("Environment variables loaded.")


# 2. Initialize the language model
llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0
)


# 3. Load the PDF document
pdf_path = os.path.join("Docs", "fabric-onelake.pdf")

if not os.path.exists(pdf_path):
    raise FileNotFoundError(
        f"PDF not found at: {pdf_path}. "
        "Check the file path and filename."
    )

loader = PyPDFLoader(pdf_path)
docs = loader.load()

print(f"Loaded {len(docs)} pages from the PDF.")


# 4. Add metadata to the documents
for doc in docs:
    doc.metadata["source"] = "fabric-onelake.pdf"
    doc.metadata["developer"] = "Microsoft"

print("Metadata added.")


# 5. Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(docs)

print(f"Created {len(chunks)} chunks.")


# 6. Initialize the embedding model
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# 7. Create and persist the Chroma vector database
vector_dir = os.path.join(".", "Vector")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=vector_dir,
    collection_name="onelake_documents"
)

print(f"Vector database saved to: {os.path.abspath(vector_dir)}")


# 8. Perform semantic search
query = "Why do we need OneLake?"

results = vectorstore.similarity_search(
    query,
    k=3
)

print("\nSEMANTIC SEARCH RESULTS")

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


# 9. Reload the persisted vector database
vectorstore_persist = Chroma(
    persist_directory=vector_dir,
    embedding_function=embedding_model,
    collection_name="onelake_documents"
)

print("\nPersisted vector database reloaded successfully.")


# 10. Search the reloaded database
query = "What are the benefits of OneLake?"

persisted_results = vectorstore_persist.similarity_search(
    query,
    k=3
)

print("\nPERSISTED DATABASE SEARCH RESULTS")

for i, doc in enumerate(persisted_results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)
